In [19]:
import pyspark
import pandas 
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/18 10:17:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2026-01.parquet



7[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87[https://d37ci6vzurychx.cloudfr]87[Files: 0  Bytes: 0  [0 B/s] Re]87Saving 'fhvhv_tripdata_2026-01.parquet'
87fhvhv_tripdata_2026-   0% [>                             ]  348.29K    --.-KB/s87[Files: 0  Bytes: 0  [0 B/s] Re]87fhvhv_tripdata_2026-   0% [>                             ]    1.45M    1.11MB/s87[Files: 0  Bytes: 0  [0 B/s] Re]87fhvhv_tripdata_2026-   0% [>                             ]    2.46M    1.06MB/s87[Files: 0  Bytes: 0  [0 B/s] Re]87fhvhv_tripdata_2026-   0% [>                             ]    3.38M    1.01MB/s87[Files: 0  Bytes: 0  [0 B/s] Re]87[Files: 0  Bytes: 0  [0 B/s] Re]87fhvhv_tripdata_2026-   0% [>                             ]    3.98M  932.46KB/s87[Files: 0  Bytes: 0  [0 B/s] Re]87fhvhv_tripdata_2026-   0% [>     

In [4]:
df = spark.read.parquet('fhvhv_tripdata_2026-01.parquet')

In [5]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_

In [6]:
df.show(5)

[Stage 1:>                                                          (0 + 1) / 1]

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+------------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|cbd_congestion_fee|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+-------

In [7]:
df = df.repartition(24)

In [8]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003') \
  .show(5)

[Stage 2:=============================>                             (2 + 2) / 4]

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2026-01-06 22:25:52|2026-01-06 22:39:55|          97|         225|
|2026-01-08 16:32:21|2026-01-08 16:54:56|         235|          47|
|2026-01-04 15:41:04|2026-01-04 16:27:28|          80|         132|
|2026-01-07 17:38:56|2026-01-07 17:52:47|          13|          68|
|2026-01-03 15:13:43|2026-01-03 15:57:48|         168|         132|
+-------------------+-------------------+------------+------------+
only showing top 5 rows


In [18]:
from pyspark.sql import functions as F
from pyspark.sql import types

In [12]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [13]:
crazy_stuff('B02884')

's/b44'

In [21]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [22]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date','dropoff_date','PULocationID','DOLocationID') \
    .show()

[Stage 8:>                                                          (0 + 1) / 1]

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/d4c| 2026-01-06|  2026-01-06|         161|         116|
|  e/d4c| 2026-01-02|  2026-01-02|          22|          22|
|  e/d4c| 2026-01-02|  2026-01-02|         130|         205|
|  e/d4c| 2026-01-08|  2026-01-08|         164|          72|
|  e/d4c| 2026-01-01|  2026-01-01|           6|         265|
|  e/d4c| 2026-01-08|  2026-01-08|         229|          68|
|  e/d4c| 2026-01-05|  2026-01-05|         133|         138|
|  e/d4c| 2026-01-08|  2026-01-08|         169|         129|
|  e/d4c| 2026-01-05|  2026-01-05|         166|          41|
|  e/d4c| 2026-01-03|  2026-01-03|          71|          91|
|  e/d4c| 2026-01-06|  2026-01-06|         177|         155|
|  e/d4c| 2026-01-02|  2026-01-02|         190|         226|
|  e/d4c| 2026-01-03|  2026-01-03|         161|         249|
|  e/d4c| 2026-01-06|  2